# 04 베이스라인: core 13개 × (로지스틱 회귀, RBF-SVM)

- 평가: `src/evaluate.py`의 중첩 CV (바깥 5-fold × 안쪽 4-fold, seed 5개)
- 비교 1: 단순 모델(로지스틱 회귀) vs 비선형 모델(RBF-SVM)
- 비교 2: 정직한 점수 vs 누수 버전 점수 (임계값을 점수 낸 데이터로 고르면 얼마나 부풀려지나)
- 바닥 점수: 전부 고장 0.556, RMS 관리도 0.080 (01 노트북)

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import fbeta_score

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.data import load_dataset
from src.features import feature_table
from src.evaluate import nested_cv, best_f2_threshold

X, y, names = load_dataset(ROOT / "data" / "TeamProject_train_변경")
F = feature_table(X).to_numpy()
groups = np.arange(len(y))  # 파일 하나 = 그룹 하나 (나중 슬라이싱 대비)

# 호출할 때마다 새 모델을 만든다 → fold마다 처음부터 학습
MODELS = {
    "로지스틱 회귀": lambda: make_pipeline(RobustScaler(), LogisticRegression(class_weight="balanced", max_iter=5000)),
    "RBF-SVM": lambda: make_pipeline(RobustScaler(), SVC(kernel="rbf", class_weight="balanced")),
}

## 1. 정직한 점수 (중첩 CV)

In [ ]:
results = {name: nested_cv(F, y, groups, make) for name, make in MODELS.items()}

rows = []
for name, r in results.items():
    rows.append({
        "모델": name,
        "F2 평균": r["f2"].mean(), "F2 표준편차": r["f2"].std(),
        "재현율": r["recall"].mean(), "정밀도": r["precision"].mean(),
        "놓친 고장(평균)": r["miss_counts"].sum() / len(r["f2"]),
        "오경보(평균)": r["fp_counts"].sum() / len(r["f2"]),
    })
print(pd.DataFrame(rows).round(3).to_string(index=False))
for name, r in results.items():
    print(f"\n{name} seed별 F2:", r["f2"].round(3))

## 2. 자주 틀리는 파일 (seed 5개 중 3번 이상)

In [ ]:
names_arr = np.array(names)
for name, r in results.items():
    print(f"{name}")
    print("  놓친 고장:", names_arr[r["miss_counts"] >= 3].tolist())
    print("  오경보 정상:", names_arr[r["fp_counts"] >= 3].tolist())

## 3. 누수 버전과 비교

누수 버전: 바깥 CV로 OOF 점수를 만든 뒤, **같은 OOF 점수**로 임계값을 고르고 그 OOF로 F₂를 계산한다.
임계값이 평가 파일의 정답을 보고 정해지므로 점수가 부풀려진다. 정직한 점수보다 높게 나와야 정상이다.

In [ ]:
def leaky_f2(make_model, seeds=range(5), n_splits=5):
    f2s = []
    for seed in seeds:
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        oof = np.empty(len(y))
        for tr, te in cv.split(F, y, groups):
            m = make_model().fit(F[tr], y[tr])
            oof[te] = m.decision_function(F[te])
        t = best_f2_threshold(y, oof)              # 평가할 데이터의 정답으로 임계값 선택 (누수)
        f2s.append(fbeta_score(y, (oof >= t).astype(int), beta=2))
    return np.array(f2s)

rows = []
for name, make in MODELS.items():
    leak = leaky_f2(make)
    rows.append({"모델": name, "정직한 F2": results[name]["f2"].mean(), "누수 F2": leak.mean(),
                 "차이": leak.mean() - results[name]["f2"].mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))